# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [5]:
# Initialization

# load_dotenv(override=True)

# openai_api_key = os.getenv('OPENAI_API_KEY')
# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
MODEL = "gpt-oss:20b"
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [6]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [7]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [8]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city: str) -> str:
    """
    Get the price of a ticket to a specified city.
    Args:
        destination_city (str): The name of the destination city.
    Returns:
    """
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [47]:
import inspect

tools = {}

def register_tool(func:callable, description:str):
    parameters = {}
    sig = inspect.signature(func)
    for param_name, param in sig.parameters.items():
        parameters[param_name] = {"type": param.annotation.__name__ if param.annotation != inspect.Parameter.empty else "str", "description": get_description(func.__doc__, param_name)}
    tools[func.__name__] = {"function": func,
                             "json": {
                                "name": func.__name__,
                    "description": description,
                    "parameters": {
                        "type": "object",
                        "properties": parameters,
                    },
                    "required": [
                        name for name, param in sig.parameters.items()
                        if param.default is inspect.Parameter.empty 
                        and param.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD)
                    ],
                    "additional_properties": False,
                    }
                }

def get_description(docstring, param_name):
    """A naive way to extract parameter descriptions from a docstring."""
    if not docstring:
        return ""
    lines = docstring.splitlines()
    for line in lines:
        line = line.strip()
        if param_name in line and ':' in line:
            return line.partition(':')[2].strip()
    return ""

In [48]:
sig = inspect.signature(get_ticket_price)
for param_name, param in sig.parameters.items():
    print(param.annotation)

<class 'str'>


In [18]:
register_tool(get_ticket_price, "Get the price of a ticket to a specified city.")
tools

{'get_ticket_price': {'function': <function __main__.get_ticket_price(destination_city: str) -> str>,
  'json': '{"name": "get_ticket_price", "description": "Get the price of a ticket to a specified city.", "parameters": {"type": "object", "properties": {"destination_city": {"type": "str", "description": "The name of the destination city."}}}, "required": ["destination_city"], "additional_properties": false}'}}

In [ ]:
get_ticket_price("London")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [ ]:
tools

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=[{"type": "function", "function": t["json"]} for t in tools])

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = ollama.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
# We have to write that function handle_tool_call:

# def handle_tool_call(message):
#     tool_call = message.tool_calls[0]
#     if tool_call.function.name == "get_ticket_price":
#         arguments = json.loads(tool_call.function.arguments)
#         city = arguments.get('destination_city')
#         price_details = get_ticket_price(city)
#         response = {
#             "role": "tool",
#             "content": price_details,
#             "tool_call_id": tool_call.id
#         }
#     return response


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [51]:
# def handle_tool_calls(message):
#     responses = []
#     for tool_call in message.tool_calls:
#         if tool_call.function.name == "get_ticket_price":
#             arguments = json.loads(tool_call.function.arguments)
#             city = arguments.get('destination_city')
#             price_details = get_ticket_price(city)
#             responses.append({
#                 "role": "tool",
#                 "content": price_details,
#                 "tool_call_id": tool_call.id
#             })
#     return responses

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        print(f"Got tool call {tool_call}")
        func = tools.get(tool_call.function.name, {})
        if func:
            arguments = json.loads(tool_call.function.arguments)
            print(f"Calling function {func['function']} with arguments {arguments}")
            result = func['function'](**arguments)
            print(f"Got result {result} from tool call")
            responses.append({
                "role": "tool",
                "content": result or "",
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [36]:
def chat(message, history):
    tools_for_model = [{"type": "function", "function": tools[t]["json"]} for t in tools]
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools_for_model)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools_for_model)
    
    return response.choices[0].message.content

In [ ]:
#[{"type": "function", "function": t["json"]} for t in tools]


[{'type': 'function',
  'function': '{"name": "get_ticket_price", "description": "Get the price of a ticket to a specified city.", "parameters": {"type": "object", "properties": {"city": {"type": "str", "description": ""}}}, "required": ["city"], "additional_properties": false}'},
 {'type': 'function',
  'function': '{"name": "set_ticket_price", "description": "Set the price of a ticket for a specified city.", "parameters": {"type": "object", "properties": {"city": {"type": "str", "description": ""}, "price": {"type": "str", "description": ""}}}, "required": ["city", "price"], "additional_properties": false}'}]

In [21]:
import sqlite3


In [22]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [46]:
def get_ticket_price(city:str) -> str:
    """
    Get the price of a ticket to a specified city.
    Args:
        destination_city (str): The name of the destination city.
    Returns:
    """
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [24]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [45]:
def set_ticket_price(city:str, price:float):
    """Set the price of a ticket to a specified city.
    Args:
        city (str): The name of the city.
        price (float): The price of the ticket.
    Returns:
        None
    """
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [43]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [44]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [49]:
tools = {}
register_tool(get_ticket_price, "Get the price of a ticket to a specified city.")
register_tool(set_ticket_price, "Set the price of a ticket for a specified city.")

In [52]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Got tool call ChatCompletionMessageFunctionToolCall(id='call_04aqok1x', function=Function(arguments='{"city":"appleton","price":"800"}', name='set_ticket_price'), type='function', index=0)
Calling function <function set_ticket_price at 0x000002ABED06BC40> with arguments {'city': 'appleton', 'price': '800'}
Got result None from tool call
Got tool call ChatCompletionMessageFunctionToolCall(id='call_yin04spv', function=Function(arguments='{"city":"Appleton"}', name='get_ticket_price'), type='function', index=0)
Calling function <function get_ticket_price at 0x000002ABED437740> with arguments {'city': 'Appleton'}
DATABASE TOOL CALLED: Getting price for Appleton
Got result Ticket price to Appleton is $800.0 from tool call


## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>